# Visualization & Evaluation

This notebook assumes the three companion modules are importable from the same directory:
- `network_init.py`  — model + data setup  
- `train_attack.py`  — UAP generation  
- `train_defense.py` — PRN denoiser  

Run the cells top-to-bottom after training has completed (or after loading saved weights/artifacts).


## 1. Setup

In [ ]:
import os
os.chdir('face-attribute-prediction')  # run from repo root


In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, accuracy_score

from network_init import setup
from train_defense import PerturbationRectifyingNetwork


In [ ]:
# ── Load model, dataloaders, attribute names ────────────────────────────
model, train_loader, val_loader, test_loader, attributes = setup()
print("Attributes:", attributes)


In [ ]:
# ── Load pre-trained UAP and defense ───────────────────────────────────
TARGET_IDX = 20          # 20 = 'Male'  |  15 = 'Eyeglasses'
CLASS_NAMES = ['Female', 'Male']

uap_noise   = torch.load('targeted_uap_noise_gender_5epochs_10xi.pt')
defense_net = torch.load('defense_gender_5epochs_10xi.pt')
defense_net = defense_net.cuda().eval()


## 2. Per-Attribute Accuracy Table

Evaluate clean / attacked / defended accuracy across **all 40 attributes**.


In [ ]:
def evaluate_per_attribute(model, loader, attributes, noise=None, defense=None):
    """Return a DataFrame with per-attribute accuracy for clean / perturbed / denoised inputs."""
    device_ids  = list(range(torch.cuda.device_count()))
    main_device = torch.device('cuda:0')

    if isinstance(model, torch.nn.DataParallel):
        model = model.module
    model = model.to(main_device)
    model = torch.nn.DataParallel(model, device_ids=device_ids, output_device=main_device)
    model.eval()

    n_attrs = len(attributes)
    correct_orig     = torch.zeros(n_attrs)
    correct_adv      = torch.zeros(n_attrs)
    correct_denoised = torch.zeros(n_attrs)
    total = 0

    with torch.no_grad():
        for images, targets in loader:
            images  = images.to(main_device)
            targets = targets.to(main_device)

            out = torch.stack(model(images)).argmax(dim=2)
            correct_orig += (out.t() == targets).sum(dim=0).cpu()
            total += images.size(0)

            if noise is not None:
                adv = torch.clamp(images + noise, 0, 1)
                out_adv = torch.stack(model(adv)).argmax(dim=2)
                correct_adv += (out_adv.t() == targets).sum(dim=0).cpu()

                if defense is not None:
                    den = defense(adv)
                    out_den = torch.stack(model(den)).argmax(dim=2)
                    correct_denoised += (out_den.t() == targets).sum(dim=0).cpu()

            del images, targets
            torch.cuda.empty_cache()

    acc_orig = correct_orig / total * 100
    data     = [attributes, acc_orig]
    cols     = ['Attribute', 'Original']

    if noise is not None:
        acc_adv = correct_adv / total * 100
        data.append(acc_adv)
        cols.append('Perturbed')
        if defense is not None:
            acc_den = correct_denoised / total * 100
            data.append(acc_den)
            cols.append('Denoised')

    df = pd.DataFrame(np.array(data).T, columns=cols).sort_values('Original', ascending=False)
    print(f"Mean original  accuracy: {acc_orig.mean():.2f}%")
    if noise is not None:
        print(f"Mean perturbed accuracy: {acc_adv.mean():.2f}%")
    if noise is not None and defense is not None:
        print(f"Mean denoised  accuracy: {acc_den.mean():.2f}%")
    return df


In [ ]:
results_df = evaluate_per_attribute(model, test_loader, attributes, uap_noise, defense_net)
results_df


## 3. Confusion Matrices — Target Attribute

Side-by-side confusion matrices for the original, attacked, and defended predictions
on the target attribute.


In [ ]:
def plot_confusion_matrix(model, loader, attribute_idx, noise, defense, class_names):
    """Plot three confusion matrices: Original | Attacked | Defended."""
    true_labels, preds_orig, preds_adv, preds_den = [], [], [], []

    with torch.no_grad():
        for images, labels in loader:
            images = images.cuda()
            labels = labels.cuda()

            def _predict(imgs):
                out = torch.stack(model(imgs)).permute(1, 0, 2)
                return torch.max(out[:, attribute_idx, :], 1)[1]

            preds_orig.append(_predict(images).cpu().numpy())

            adv = torch.clamp(images + noise, 0, 1)
            preds_adv.append(_predict(adv).cpu().numpy())

            den = defense(adv)
            preds_den.append(_predict(den).cpu().numpy())

            true_labels.append(labels[:, attribute_idx].cpu().numpy())

    y_true = np.concatenate(true_labels)
    y_orig = np.concatenate(preds_orig)
    y_adv  = np.concatenate(preds_adv)
    y_den  = np.concatenate(preds_den)

    acc_orig = accuracy_score(y_true, y_orig) * 100
    acc_adv  = accuracy_score(y_true, y_adv)  * 100
    acc_den  = accuracy_score(y_true, y_den)  * 100

    cm_orig = confusion_matrix(y_true, y_orig, normalize='all')
    cm_adv  = confusion_matrix(y_true, y_adv,  normalize='all')
    cm_den  = confusion_matrix(y_true, y_den,  normalize='all')

    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    FONT = 16

    configs = [
        (cm_orig, 'Blues',  f'Original\nAccuracy: {acc_orig:.2f}%'),
        (cm_adv,  'Reds',   f'Attacked\nAccuracy: {acc_adv:.2f}%'),
        (cm_den,  'Greens', f'Defended\nAccuracy: {acc_den:.2f}%'),
    ]
    for ax, (cm, cmap, title) in zip(axes, configs):
        sns.heatmap(cm, annot=True, fmt='.2%', cmap=cmap, ax=ax,
                    xticklabels=class_names, yticklabels=class_names,
                    annot_kws={'size': FONT})
        ax.set_title(title, fontsize=14)
        ax.set_xlabel('Predicted', fontsize=12)
        ax.set_ylabel('True', fontsize=12)
        ax.tick_params(labelsize=12)

    plt.tight_layout()
    plt.show()


In [ ]:
plot_confusion_matrix(model, test_loader, TARGET_IDX, uap_noise, defense_net, CLASS_NAMES)


## 4. Sample Image: Original → Attacked → Defended

Visualise a single image from the validation set to see the effect of the
UAP and the denoiser side-by-side.


In [ ]:
from network_init import build_dataloaders
_, val_dataset_raw, _ = build_dataloaders.__module__, None, None
# Rebuild val_dataset without the DataLoader wrapper for index access
from celeba import CelebA
import torchvision.transforms as T
val_dataset = CelebA('./data/celeba', 'val_40_att_list.txt', T.Compose([T.ToTensor()]))


In [ ]:
SAMPLE_IDX = 7

img_tensor, _ = val_dataset[SAMPLE_IDX]
img_tensor = img_tensor.unsqueeze(0).cuda()

with torch.no_grad():
    noisy_tensor    = torch.clamp(img_tensor + uap_noise, 0, 1)
    denoised_tensor = defense_net(noisy_tensor)

def _pred(t):
    return torch.argmax(model(t)[TARGET_IDX][0]).item()

labels_display = {
    'Original':  CLASS_NAMES[_pred(img_tensor)],
    'Attacked':  CLASS_NAMES[_pred(noisy_tensor)],
    'Defended':  CLASS_NAMES[_pred(denoised_tensor)],
}

def _to_np(t):
    return t.squeeze().permute(1, 2, 0).cpu().numpy()

fig, axes = plt.subplots(1, 3, figsize=(14, 6))
for ax, (title, img) in zip(axes, [
    (f"Original\n({labels_display['Original']})",  _to_np(img_tensor)),
    (f"With UAP filter\n({labels_display['Attacked']})", _to_np(noisy_tensor)),
    (f"Denoised\n({labels_display['Defended']})",  _to_np(denoised_tensor)),
]):
    ax.imshow(img)
    ax.set_title(title, fontsize=13)
    ax.axis('off')

plt.tight_layout()
plt.show()


## 5. UAP Filter Pattern

Amplify and display the raw UAP noise to make its spatial structure visible.


In [ ]:
def show_filter_pattern(noise_tensor: torch.Tensor) -> None:
    """Normalise the noise to [0,1] and display it as an RGB image."""
    noise = noise_tensor.cpu().squeeze()
    lo, hi = noise.min(), noise.max()
    if hi - lo > 0:
        amplified = ((noise - lo) / (hi - lo)).permute(1, 2, 0)
        plt.figure(figsize=(6, 6))
        plt.imshow(amplified.numpy())
        plt.title(f'UAP Filter (amplified)\nTrue range: [{lo:.4f}, {hi:.4f}]')
        plt.axis('off')
        plt.show()
    else:
        print("Noise tensor is all zeros — nothing to display.")

show_filter_pattern(uap_noise)
